# Quasi-Trefftz DG method for the acoustic wave equation

In [1]:
from ngsolve.TensorProductTools import *
from trefftzngs import *
#from ngsolve import * # this is also performed by previous command 
from netgen.geom2d import unit_square

We construct a space-time Cartesian in time mesh by

In [2]:
meshx = Mesh(unit_square.GenerateMesh(maxh=0.2))
mesht = Mesh(SegMesh(int(1/0.2),0,1,periodic=False) )
mesh = TensorProdMesh(meshx,mesht)

Using the Airy function that solves, which fulfills $$\mathrm{Airy}''(x)=x\mathrm{Airy}(x),$$
we construct the solution to the wave equation with $$-\Delta U + c^{-2}\partial_{tt}U=0$$ wavespeed $$c=\frac{1}{\sqrt{1+x+y}}.$$
The time coordinate is built into the mesh as the z-Axis.

In [3]:
bdd = CoefficientFunction((
        airy((-x-y-1))*cos(sqrt(2)*z),
        -airyp((-x-y-1))*cos(sqrt(2)*z),
        -airyp((-x-y-1))*cos(sqrt(2)*z),
        -airy((-x-y-1))*sin(sqrt(2)*z)*sqrt(2)
    ))

c=CoefficientFunction(1/sqrt(1+x+y))

We define the quasi-Trefftz space for the second-order wave equation on a mesh element $K\in\mathcal T_h$ as
\begin{equation}\label{eq:QU}
\mathbb Q^p(K):=\big\{
f\in\mathbb{P}^p(K) \mid D^i\square_G f(x_K,t_K)=0,\ \forall i\in \mathbb N^{n+1}_0, |i|<p-1
\big\},
\qquad p\in \mathbb N.
\end{equation}


We use the quasi-Trefftz space by setting "useqt=True"

In [4]:
fes = trefftzfespace(mesh, order=4, useqt=True)
fes.SetWavespeed(c)

Defining the followning DG averages and jumps for 
$$U\in \mathbb Q^p,\quad \sigma=-\nabla U,\ v=\partial_t U$$

In [5]:
D = mesh.dim - 1
U = fes.TrialFunction()
V = fes.TestFunction()
gU = grad(U)
gV = grad(V)

v = gU[D]
sig = CoefficientFunction(tuple([-gU[i] for i in  range(D)]))
w = gV[D]
tau = CoefficientFunction(tuple([-gV[i] for i in  range(D)]))

vo = gU.Other()[D]
sigo = CoefficientFunction(tuple([-gU.Other()[i] for i in  range(D)]))
wo = gV.Other()[D]
tauo = CoefficientFunction(tuple([-gV.Other()[i] for i in  range(D)]))

h = specialcf.mesh_size
n = specialcf.normal(D+1)
n_t = n[D]/Norm(n)
n_x = CoefficientFunction( tuple([n[i]/Norm(n) for i in  range(D)]) )

mean_v = 0.5*(v+vo)
mean_w = 0.5*(w+wo)
mean_sig = 0.5*(sig+sigo)
mean_tau = 0.5*(tau+tauo)

jump_vx = ( v - vo ) * n_x
jump_wx = ( w - wo ) * n_x
jump_sigx = (( sig - sigo ) * n_x)
jump_taux = (( tau - tauo ) * n_x)

jump_wt = ( w - wo ) * n_t
jump_taut = ( tau - tauo ) * n_t

jump_Ut = (U - U.Other()) * n_t
jump_Vt = (V - V.Other()) * n_t

timelike = n_x*n_x # n_t=0
spacelike = n_t**2 # n_x=0

Initial and boundary conditions:

In [6]:
v0=bdd[3]
sig0=CoefficientFunction(tuple([-bdd[i] for i in  range(1,3)]))
gD=v0
U0=bdd[0]

We consider the following variational formulation:
$$
\begin{align}
\text{Seek}&\quad (v_{hp},\sigma_{hp})\in V_p (\mathcal{T}_h)\nonumber\\
\text{such that}&\quad
\mathcal{A}(v_{hp},\sigma_{hp}; w ,\tau )=\ell( w ,\tau )\quad 
\forall ( w ,\tau )\in V_p (\mathcal{T}_h), 
\nonumber\\
\text{where}&\nonumber\\
\mathcal{A}(v_{hp},&\sigma_{hp}; w ,\tau ):=
-\sum_{K\in\mathcal{T}_h} \int_K\bigg(v\Big(\nabla\cdot\tau+c^{-2}\partial_t w \Big) +\sigma_{hp}\cdot\Big(\partial_t \tau +\nabla w  \Big)\bigg)
\nonumber\\
+&\int_{F^{space}}\big(c^{-2}v_{hp}^-\{\{{w}\}\}_t+\sigma_{hp}^-\cdot\{\{{\tau}\}\}_t+v_{hp}^-\{\{{\tau}\}\}_N+\sigma_{hp}^-\cdot\{\{{w}\}\}_N\big)
\nonumber
\\
+&\int_{F^{time}}\!\! \big( [[{v_{hp}}]]\{\{{\tau }\}\}_N+[[{\sigma_{hp}}]]\cdot\{\{{ w }\}\}_N
+\alpha\{\{{v_{hp}}\}\}_N\cdot\{\{{ w }_N+ \beta\{\{{\sigma_{hp}}\}\}_N\{\{{\tau }\}\}_N
\big)
\nonumber\\
+&\int_{F^T} (c^{-2}v_{hp}  w +\sigma_{hp} \cdot\tau )
+\int_{F^D} \big(\sigma\cdot n_\Omega^x\, w +\alpha v_{hp} w   \big) 
\nonumber\\
\ell( w ,&\tau ):=
\int_{F^O} ( c^{-2}v_0 w  +\sigma_0\cdot \tau )
+\int_{F^D} g_D\big(\alpha  w -\tau\cdot n_\Omega^x\big)
\nonumber
\end{align}
$$
    + correction terms to recover the second order solution

In [7]:
a = BilinearForm(fes)
HV = V.Operator("hesse")
a += SymbolicBFI(  -v*(-sum([HV[i*(D+2)] for i in range(D)]) + pow(c,-2)*HV[(D+1)*(D+1)-1]) )
#space like faces, w/o x jump
a += SymbolicBFI( spacelike * ( IfPos(n_t,v,vo)*(pow(c,-2)*jump_wt) + IfPos(n_t,sig,sigo)*(jump_taut) ), VOL, skeleton=True )
#time like faces
a += SymbolicBFI( timelike * ( mean_v*jump_taux + mean_sig*jump_wx), VOL, skeleton=True )        #t=T (or *x)
a += SymbolicBFI( ( pow(c,-2)*v*w + sig*tau ), BND, definedon=fes.mesh.Boundaries("outflow"), skeleton=True)
#dirichlet boundary 'timelike'
a += SymbolicBFI( ( sig*n_x*w), BND, definedon=fes.mesh.Boundaries("dirichlet"), skeleton=True)
#correction term to recover sol of second order system
a += SymbolicBFI( spacelike * ( jump_Ut*jump_Vt ), VOL, skeleton=True )
a += SymbolicBFI( ( U*V ), BND, definedon=fes.mesh.Boundaries("inflow"), skeleton=True )
a.Assemble()

f = LinearForm(fes)
f += SymbolicLFI( ( pow(c,-2)*v0*w + sig0*tau ), BND, definedon=fes.mesh.Boundaries("inflow"), skeleton=True) #t=0 (or *(1-x))
f += SymbolicLFI( ( gD * (-tau*n_x) ), BND, definedon=fes.mesh.Boundaries("dirichlet"), skeleton=True) #dirichlet boundary 'timelike'
f += SymbolicLFI( U0*V, BND, definedon=fes.mesh.Boundaries("inflow"),  skeleton=True ) #rhs correction term to recover sol of second order system
f.Assemble()

Solve

In [8]:
gfu = GridFunction(fes, name="uDG")
gfu.vec.data = a.mat.Inverse()*f.vec

Error at final time $T=1$

In [9]:
sqrt(Integrate((BoundaryFromVolumeCF(grad(gfu)[D]) - bdd[D+1])**2/c**2+sum((BoundaryFromVolumeCF(grad(gfu)[i]) - bdd[i+1])**2 for i in range(D)), mesh, definedon=fes.mesh.Boundaries("outflow")))

2.8403926701282696e-05